This notebook must be executed inside the FINN container provided by Xilinx.

Follow the steps of the [FINN docks](https://finn.readthedocs.io/en/latest/), section [Quickstart](https://finn.readthedocs.io/en/latest/getting_started.html#running-finn-in-docker), to download, build and verify the container installation.

You must also move your project folder to inside the same folder the repository is located.

To start the container, go to the folder where the repo was installed and run $ ./run-docker.sh notebook

If you are using vscode, you can select the notebook kernel inside the container to run the code.

This notebook is strongly based on Xilinx [tfc_end2end_example.ipynb](https://github.com/Xilinx/finn/blob/main/notebooks/end2end_example/bnn-pynq/tfc_end2end_example.ipynb) notebook and 0BAB1 [2_finn_hardware_layers.ipynb](https://github.com/0BAB1/tutorial-snippets/blob/main/8%20Python%20to%20FPGA/2_finn_hardware_layers.ipynb) notebook.

Check them out for deeper instructions.

# Setup Python Paths for Libraries

In [1]:
import os
import sys

# Correct the path where the environment starts to be the same folder of the notebook, so that the imports work correctly.
# When starting the container with the notebook server, the script hardcodes the Jupyter server to start inside the ./notebooks folder.
os.chdir('../QFast-SCNN_with_Brevitas_and_FINN/finn_environment')

# Add the train_environment directory to the system path to allow imports from there
sys.path.append(os.path.abspath('../train_environment'))
print(sys.path)

['/usr/lib/python310.zip', '/usr/lib/python3.10', '/usr/lib/python3.10/lib-dynload', '', '/tmp/home_dir/.local/lib/python3.10/site-packages', '/home/jose-vitor/finn-repo/deps/qonnx/src', '/home/jose-vitor/finn-repo/deps/finn-experimental/src', '/home/jose-vitor/finn-repo/deps/brevitas/src', '/home/jose-vitor/finn-repo/deps/pyverilator', '/home/jose-vitor/finn-repo/src', '/usr/local/lib/python3.10/dist-packages', '/workspace/src/dataset-loading', '/usr/lib/python3/dist-packages', '/home/jose-vitor/finn-repo/QFast-SCNN_with_Brevitas_and_FINN/train_environment']


# Setup Model for FINN

* Tidy up (and also after EACH step)
* Pre (data feed) / Post proc (top k)
* Model streamlining (Main step) + smaller example
* Model HW Layers (Generates Matrix Vector Activation Units for fc layers)
* Model data flow partitions (Generate a sub-graph for all HW convertible nodes)
* Specialize layer, ready for hw conversion (generates hls for the dataflow partition node)

### Tidy Up

In [43]:
from finn.util.visualization import showSrc, showInNetron
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.core.modelwrapper import ModelWrapper
from pathlib import Path

# Setup Path
FINN_BIT_WIDTH = 8
onnx_path = f'../onnx/quant_model_{FINN_BIT_WIDTH}_bits.onnx'
onnx_name = Path(onnx_path).stem

model = ModelWrapper(onnx_path)

# TIDY UP
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())

tidy_path = f'./tidy_onnx/{onnx_name}_tidy.onnx'
Path(tidy_path).parent.mkdir(parents=True, exist_ok=True)
model.save(tidy_path)

showSrc(InferShapes)

/home/jose-vitor/finn-repo/deps/qonnx/src/qonnx/util/onnx.py:40: DeprecationWarning: `mapping.TENSOR_TYPE_TO_NP_TYPE` is now deprecated and will be removed in a future release.To silence this warning, please use `helper.tensor_dtype_to_np_dtype` instead.
  return np.zeros(dims, dtype=onnx.mapping.TENSOR_TYPE_TO_NP_TYPE[vi.type.tensor_type.elem_type])


class InferShapes(Transformation):
    """Ensure every tensor in the model has a specified shape (ValueInfo)."""

    def apply(self, model):
        # hide your riches!
        hidden_ops = _hide_finn_ops(model)
        # call regular ONNX shape inference
        model = ModelWrapper(si.infer_shapes(model.model))
        # bring back hidden ops
        _restore_finn_ops(model, hidden_ops)
        return (model, False)



In [3]:
showInNetron(tidy_path)

Serving './tidy_onnx/quant_model_8_bits_tidy.onnx' at http://0.0.0.0:8081


### Pre processing

FINN model expects UINT8 input. According to Xilinx, this is highly beneficial for performance, because you can directly input raw data to the model, instead of relying on CPU for pre processing.

The the QFast-SCNN model exported to QONNX has the pre processing layers integrated in the Pytorch model, so the expected input is already from 0 to 255.

If the target model was trained with tensor inputs different than [0, 255], like the standard torch.Tensor [0, 1] or tensors with Imagenet normalization, you have two main options to follow:
* Modify your Pytorch model only for the QONNX export, integrating the pre processing inside the model (the option I have chosen for QFast-SCNN).
* Add the preprocessing layers in the QONNX model, following the "Adding Pre- and Postprocessing" section of Xilinx [tfc_end2end_example.ipynb](https://github.com/Xilinx/finn/blob/main/notebooks/end2end_example/bnn-pynq/tfc_end2end_example.ipynb) notebook.

In [44]:
from finn.util.pytorch import ToTensor
from qonnx.transformation.merge_onnx_models import MergeONNXModels
from qonnx.core.datatype import DataType
from brevitas.export import export_qonnx
from qonnx.util.cleanup import cleanup as qonnx_cleanup
import torch
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN

# PRE PROC : NONE
model = ModelWrapper(tidy_path)

# add input annotation: UINT8 is what we will feed the model during inference
global_inp_name = model.graph.input[0].name
model.set_tensor_datatype(global_inp_name, DataType["UINT8"])

# Save the preprocessed model
preproc_path = f'./preproc_onnx/{onnx_name}_preproc.onnx'
Path(preproc_path).parent.mkdir(parents=True, exist_ok=True)
model.save(preproc_path)

In [4]:
showInNetron(preproc_path)

Serving './preproc_onnx/quant_model_8_bits_preproc.onnx' at http://0.0.0.0:8081


### Compare the Outputs of Pytorch Model and QONNX Model

Optional but highly recommended step. The Pytorch model will be loaded with the "finn" mode so the test input of this model is the same as the QONNX model.

In [22]:
import torch
from torchvision import transforms
from torchvision.datasets import Cityscapes
from my_finn_utils import load_state_dict, generate_cityscapes_labels, IdToTrainIdTransform
import models.QFastSCNN as qfscnn
from config import NUM_CLASSES, DATA_PATH

lable_conversion, id_names = generate_cityscapes_labels()

# Defining the Cityscapes validation dataset.
val_dataset = Cityscapes(
    root=DATA_PATH,
    split='val',
    mode='fine',
    target_type='semantic',
    transform=transforms.PILToTensor(), # Converting the PIL images to tensors, keeping the original pixel values (0-255) which is important for the quantized model that expects UINT8 inputs.
    target_transform=transforms.Compose([
        transforms.PILToTensor(), # Converting the PIL masks to tensors, keeping the original pixel values (0-255).
        IdToTrainIdTransform(lable_conversion), # Converting the original Cityscapes labels to the 19 classes used for training and evaluation, as per the Cityscapes benchmark.
    ])
)

# Importing the test image and mask
img_tensor, smnt_tensor = val_dataset[0]
img_tensor = img_tensor.unsqueeze(0) # Add batch dimension
print("Input image shape:", img_tensor.shape)
print("Input mask shape:", smnt_tensor.shape)

# Creating a Brevitas model instance and loading the quantized weights from the training phase.
brevitas_model = qfscnn.QFastSCNN(NUM_CLASSES, mode="finn")
brevitas_model = load_state_dict(brevitas_model, path="../train_environment/model_weights/quant_params/best_quant_model.pth", strict=False)
brevitas_model.eval();

Input image shape: torch.Size([1, 3, 1024, 2048])
Input mask shape: torch.Size([1, 1024, 2048])
Carregando modelo best_quant_model


In [40]:
import torch.nn.functional as F

# Run a foward pass on Brevitas model
with torch.inference_mode():
    brevitas_output = brevitas_model(img_tensor)
brevitas_output_upsampled = F.interpolate(brevitas_output, size=img_tensor.shape[2:], mode='bilinear', align_corners=False)
brevitas_output_mask = torch.softmax(brevitas_output_upsampled, dim=1).argmax(dim=1).to(torch.uint8)

# Calculate how many pixels are matching between the Brevitas output mask and the ground truth mask, and print the results.
matching_pixels = (brevitas_output_mask == smnt_tensor).sum().item()

print(f"Output shape from Brevitas model: {brevitas_output.shape}\n"
      f"Output shape after upsampling: {brevitas_output_upsampled.shape}\n"
      f"Output mask shape: {brevitas_output_mask.shape}\n"
      f"Accuracy: {(100 * matching_pixels/smnt_tensor.numel()):.2f}%\n")

Output shape from Brevitas model: torch.Size([1, 19, 128, 256])
Output shape after upsampling: torch.Size([1, 19, 1024, 2048])
Output mask shape: torch.Size([1, 1024, 2048])
Accuracy: 83.19%



In [ ]:
from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe

# Run the QONNX inference.
model = ModelWrapper(preproc_path)
input_dict = {"global_in": img_tensor.cpu().numpy()}
#input_dict = {"global_in": nph.to_array(input_tensor)}
output_dict = oxe.execute_onnx(model, input_dict)
produced_qonnx = output_dict[list(output_dict.keys())[0]]

Exception: Found unspecified tensor shapes, try infer_shapes